# Formal Stage 5 — no-anchor 6/20（seed 42）

这是冻结后的正式候选发现入口。配置从创建时即为1000步；第一次只运行到第10步，检查通过后恢复同一个run并继续到1000。所有历史diagnostic、registry和checkpoint只读，不恢复任何旧训练状态。

### 第 0 格：检查安全开关与正式冻结合同

10步检查已经通过。本格现在固定为 `MODE='resume'`、`TARGET_STEP=1000`，并已填入唯一正式run目录。不要改回new，不要修改 `PLANNED_MAX_STEPS=1000` 或run目录。

In [ ]:
import json, platform, sys
from dataclasses import asdict
from pathlib import Path
import pandas as pd

PROJECT_ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'factor_gfn').is_dir())
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
from factor_gfn.gfn import (FORMAL_STAGE5_NO_ANCHOR_CONFIG_FINGERPRINT, FORMAL_STAGE5_NO_ANCHOR_MAX_STEPS, ExhaustiveRegistry, RealRewardDataPaths, RealSearchSettings, build_frozen_stage5_no_anchor_6_20_config, create_no_anchor_real_search_runner, resume_no_anchor_real_search_runner)
from factor_gfn.gfn.diagnostic_support import progress_heartbeat

RUN_FORMAL_STAGE5=True
MODE='resume'
TARGET_STEP=1000
RESUME_RUN_DIR=PROJECT_ROOT/'runs'/'stage5_no_anchor_formal_6_20'/'c3a1c2747cbb41dbbb3f8f23e6ddddcb'
DEVICE='cuda:0'
PLANNED_MAX_STEPS=1000
CHECKPOINT_INTERVAL=10
SOURCE_DIAGNOSTIC_ROOT=PROJECT_ROOT/'runs'/'complexity_diagnostic_6_20'/'manual_diagnostic_6_20_seed42'
SOURCE_REGISTRY=SOURCE_DIAGNOSTIC_ROOT/'exhaustive_registry.sqlite3'
TARGETED_ARTIFACT=PROJECT_ROOT/'runs'/'targeted_calibration_6_20'/'targeted_logz_n17_n18_seed42'/'targeted_log_z_engineering_initialization.json'
RUN_ROOT=PROJECT_ROOT/'runs'/'stage5_no_anchor_formal_6_20'
config=build_frozen_stage5_no_anchor_6_20_config(); strata=config.resolved_strata()
assert PLANNED_MAX_STEPS==FORMAL_STAGE5_NO_ANCHOR_MAX_STEPS==config.training.max_steps
assert config.fingerprint()==FORMAL_STAGE5_NO_ANCHOR_CONFIG_FINGERPRINT
print({'enabled':RUN_FORMAL_STAGE5,'mode':MODE,'target_step':TARGET_STEP,'resume_run_dir':None if RESUME_RUN_DIR is None else str(RESUME_RUN_DIR),'planned_max_steps':PLANNED_MAX_STEPS,'device':DEVICE,'config_fingerprint':config.fingerprint(),'F':strata.feasible_node_counts,'D':strata.discovery_node_counts,'E':strata.exact_normalizer_node_counts,'L':strata.learned_normalizer_node_counts,'batch_size':config.training.batch_size,'policy_lr':config.training.learning_rate,'logZ_lr':config.training.log_z_learning_rate,'policy/logZ_max_norm':(config.training.model_gradient_clip_norm,config.training.log_z_gradient_clip_norm),'retry_budget':config.complexity.exact_node_retry_budget},flush=True)
print('10步预计约10–15分钟；1000步按当前约58秒/步估算约16小时，实际ETA由每步一行动态更新。',flush=True)

### 第 1 格：创建或恢复正式 run

`new`会加载training-only数据、重新做一次N=1/2 registry等价证明，并导入historical/N17–18初始化常数，随后创建全新的正式run；不会写入任何来源目录。`resume`只接受本Notebook的新no-anchor schema和冻结指纹。初始化可能数分钟，超过20秒会显示heartbeat。

In [ ]:
if not RUN_FORMAL_STAGE5: raise RuntimeError('安全停止：确认第0格后，将RUN_FORMAL_STAGE5=True')
if MODE not in {'new','resume'}: raise ValueError("MODE只能是'new'或'resume'")
if MODE=='new' and RESUME_RUN_DIR is not None: raise ValueError('new模式不得设置RESUME_RUN_DIR')
if MODE=='resume' and RESUME_RUN_DIR is None: raise ValueError('resume模式必须显式填写RESUME_RUN_DIR')
if not 1<=TARGET_STEP<=PLANNED_MAX_STEPS: raise ValueError('TARGET_STEP必须位于1..1000')
for required in (SOURCE_REGISTRY,TARGETED_ARTIFACT,SOURCE_DIAGNOSTIC_ROOT/'diagnostic_summary.json',SOURCE_DIAGNOSTIC_ROOT/'diagnostic_context.json'):
    if not required.is_file(): raise FileNotFoundError(required)
settings=RealSearchSettings(max_steps=PLANNED_MAX_STEPS,seed=42,checkpoint_interval=CHECKPOINT_INTERVAL,device=DEVICE,cache_max_entries=50_000,subexpression_cache_max_bytes=0,tensorboard_enabled=True,console_progress=True,run_root=RUN_ROOT)
if MODE=='new':
    print('[stage5-init] creating fresh formal run; heartbeat every20s',flush=True)
    with progress_heartbeat('formal Stage5 initialization',interval_seconds=20.0): runner=create_no_anchor_real_search_runner(settings,registry_path=SOURCE_REGISTRY,historical_diagnostic_root=SOURCE_DIAGNOSTIC_ROOT,targeted_artifact_path=TARGETED_ARTIFACT,paths=RealRewardDataPaths())
else:
    print('[stage5-init] resuming exact formal run; heartbeat every20s',flush=True)
    with progress_heartbeat('formal Stage5 resume',interval_seconds=20.0): runner=resume_no_anchor_real_search_runner(Path(RESUME_RUN_DIR),paths=RealRewardDataPaths())
assert runner.config.fingerprint()==FORMAL_STAGE5_NO_ANCHOR_CONFIG_FINGERPRINT
assert runner.trainer.reward_provider.manifest()['validation_oos_loaded'] is False
assert runner.trainer.step<TARGET_STEP<=runner.config.training.max_steps
print('[stage5-init]',{'run_id':runner.trainer.run_id,'run_dir':str(runner.run_dir),'current_step':runner.trainer.step,'optimizer_step':runner.trainer.optimizer_step,'target_step':TARGET_STEP,'checkpoint':str(runner.latest_checkpoint_path)},flush=True)

### 第 2 格：运行到TARGET_STEP

这是长训练格。每完成一个step只输出一行：step/1000、target、optimizer step、成功/跳过、valid/request、累计retry耗尽、loss、Reward、TB delta、裁剪、实际更新、entropy、learned-logZ范围、单步耗时、ETA和显存。每步原子保存latest checkpoint，且每10步保存归档checkpoint。首次只跑到10，完成后把run目录交给Codex检查。

In [ ]:
new_stats=runner.run_until(TARGET_STEP)
print('[stage5-stop]',{'run_dir':str(runner.run_dir),'current_step':runner.trainer.step,'optimizer_step':runner.trainer.optimizer_step,'target_step':TARGET_STEP,'new_steps':len(new_stats),'latest_checkpoint':str(runner.latest_checkpoint_path)},flush=True)
print('若当前是10步：停止，不要直接续跑；先把上述run目录交给Codex检查。',flush=True)

### 第 3 格：只读查看当前运行摘要并关闭registry

本格不训练，只读取已原子保存的summary/state，显示一行关键累计信息并关闭只读registry。预计数秒。正式候选、逐步指标和checkpoint都保留在run目录。

In [ ]:
state=json.loads((runner.run_dir/'run_state.json').read_text(encoding='utf-8')); summary=json.loads((runner.run_dir/'training_stats.json').read_text(encoding='utf-8'))
print('[stage5-summary]',{'status':state['status'],'current_step':state['current_step'],'optimizer_step':state['optimizer_step'],'evaluations':state['evaluation_records'],'unique_expressions':summary['unique_expressions'],'valid_reward_requests':summary['valid_reward_requests'],'total_wall_hours':summary['total_wall_seconds']/3600,'config_fingerprint':runner.config.fingerprint()},flush=True)
runner.close()
print('FORMAL_STAGE5_SEGMENT_COMPLETE',flush=True)